[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C15_Classic_Architectures_Course/05_seq2seq_attention/05_seq2seq_attention.ipynb)

# 05 · seq2seq 与注意力（纯 numpy 从零）

目标：从零实现 **encoder-decoder**、**Bahdanau 加性注意力**与 **Luong 乘性注意力**、**对齐矩阵**、**teacher forcing 与贪心解码**，并用 `assert`/对拍/数值梯度验证每一步。

路线：工具 → 加性注意力 → 乘性注意力(对拍朴素循环) → 对齐矩阵(行和=1) → 注意力可微(数值梯度) → teacher forcing vs 贪心 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊（复制/反转任务上，注意力学出对角/反对角对齐）。

> 心智模型：**注意力 = 打分 → softmax → 加权求和**。它让 decoder 每一步动态回看源序列，是 Transformer 的直系祖先。

## 0 · 工具：softmax / 数值梯度 / 对拍

沿用全课的三件套（与模块 00 一致），后面反复用。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    '''数值稳定 softmax：先减最大值再 exp。'''
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def numerical_grad(f, x, eps=1e-5):
    '''中心差分逐元素估计 df/dx。f: ndarray->标量。'''
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f(x)
        x[idx] = old - eps; fm = f(x)
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

def rel_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(1e-8, np.abs(a) + np.abs(b))))

def check_allclose(name, got, ref, atol=1e-10):
    got = np.asarray(got, float); ref = np.asarray(ref, float)
    ok = np.allclose(got, ref, atol=atol)
    print(f'[{name:<30}] allclose={ok}  max|err|={np.max(np.abs(got-ref)):.2e}')
    assert ok, f'{name} 不一致'
    return ok

s = softmax(np.array([[1000., 1001., 1002.]]))
assert np.allclose(s.sum(-1), 1.0) and not np.isnan(s).any()
print('✅ 工具就绪：softmax / numerical_grad / rel_error / check_allclose')

## 1 · encoder：把源序列压成隐藏状态序列

encoder 就是模块 03 的 RNN：逐步吃源词嵌入、更新隐藏状态。注意力时代我们**保留每一步的隐藏状态** `H = [h_1,...,h_T]`（而非只留末态）。

下面用 tanh-RNN 当 encoder，输出全部隐藏状态（供注意力回看）与末态（供 decoder 初始化）。

In [ ]:
def rnn_encoder(X, Wx, Wh, b, h0=None):
    '''X: (T, D_in) 源序列嵌入；返回 H:(T, D_h) 全部隐藏状态, h_T:(D_h,) 末态。'''
    T, D_in = X.shape
    D_h = Wh.shape[0]
    h = np.zeros(D_h) if h0 is None else h0.copy()
    H = np.zeros((T, D_h))
    for t in range(T):
        h = np.tanh(Wx @ X[t] + Wh @ h + b)   # 模块 03 的 RNN 递推
        H[t] = h
    return H, h

T, D_in, D_h = 4, 5, 6
X = rng.standard_normal((T, D_in))
Wx = rng.standard_normal((D_h, D_in)) * 0.3
Wh = rng.standard_normal((D_h, D_h)) * 0.3
b  = np.zeros(D_h)
H, hT = rnn_encoder(X, Wx, Wh, b)
print('源长度 T =', T, '| H 形状(全部隐藏态) =', H.shape, '| 末态 h_T 形状 =', hT.shape)
assert H.shape == (T, D_h) and hT.shape == (D_h,)
assert np.allclose(H[-1], hT), '末态应等于 H 的最后一行'
print('✅ encoder：保留全部隐藏状态 H（注意力要用），末态 = H[-1]')

## 2 · Bahdanau 加性注意力：打分 → softmax → 加权求和

给定 decoder 状态 `s`(D_s) 与 encoder 全部状态 `H`(T, D_e)，三步：
1. 打分（能量）`e_j = vᵀ tanh(W_s s + W_h h_j)`；
2. 权重 `α = softmax(e)`（一个源位置上的概率分布）；
3. 上下文 `c = Σ_j α_j h_j`。

**断言**：权重和为 1、形状为 (T,)、上下文形状为 (D_e,)。

In [ ]:
def bahdanau_attention(s, H, v, Ws, Wh):
    '''加性注意力。s:(D_s,)  H:(T,D_e)  v:(D_a,)  Ws:(D_a,D_s)  Wh:(D_a,D_e)。
       返回 context:(D_e,), alpha:(T,) 注意力权重。'''
    proj_s = Ws @ s                      # (D_a,)  query 投影
    proj_H = H @ Wh.T                    # (T, D_a) 每个 key 投影
    energies = np.tanh(proj_s[None, :] + proj_H) @ v   # (T,) 广播相加→tanh→投到标量
    alpha = softmax(energies)           # (T,) 注意力权重
    context = alpha @ H                  # (D_e,) 加权求和 Σ α_j h_j
    return context, alpha

D_s, D_e, D_a = 6, 6, 7
s = rng.standard_normal(D_s)
v  = rng.standard_normal(D_a)
Ws = rng.standard_normal((D_a, D_s)) * 0.5
Wh = rng.standard_normal((D_a, D_e)) * 0.5
context, alpha = bahdanau_attention(s, H, v, Ws, Wh)
print('注意力权重 alpha =', np.round(alpha, 3), '| 和 =', alpha.sum())
print('上下文 context 形状 =', context.shape)
assert alpha.shape == (T,) and context.shape == (D_e,)
assert np.allclose(alpha.sum(), 1.0), '注意力权重应和为 1'
assert (alpha >= 0).all(), '注意力权重应非负'
# 对拍：用朴素逐 j 循环算能量，应与向量化一致
e_naive = np.array([v @ np.tanh(Ws @ s + Wh @ H[j]) for j in range(T)])
check_allclose('additive energies 向量化 vs 循环', softmax(e_naive), alpha)
print('✅ Bahdanau 加性注意力：权重和=1、与朴素循环对拍一致')

## 3 · Luong 乘性注意力：dot / general，对拍朴素循环

用点积代替小前馈网打分，更高效：
- **dot**：`score = sᵀ h_j`（要求 D_s == D_e）；
- **general**：`score = sᵀ W h_j`（允许异维，含可学习 W）。

点积可写成一次矩阵乘 `E = H @ s`（dot）——这是 Transformer `QKᵀ` 的雏形。下面实现并与逐 j 循环对拍。

In [ ]:
def luong_attention(s, H, W=None):
    '''乘性注意力。W=None 用 dot(需同维)，否则 general: sᵀ W h_j。
       返回 context:(D_e,), alpha:(T,)。'''
    if W is None:
        energies = H @ s                  # (T,) dot: 每行 h_j·s
    else:
        energies = (H @ W.T) @ s          # (T,) general: (h_j^T W^T) s = s^T W h_j
    alpha = softmax(energies)
    context = alpha @ H
    return context, alpha

# dot（D_s==D_e==6）
ctx_dot, a_dot = luong_attention(s, H)
e_dot_naive = np.array([s @ H[j] for j in range(T)])
check_allclose('Luong dot 向量化 vs 循环', a_dot, softmax(e_dot_naive))
assert np.allclose(a_dot.sum(), 1.0)
# general（带 W）
Wg = rng.standard_normal((D_e, D_s)) * 0.5
ctx_gen, a_gen = luong_attention(s, H, W=Wg)
e_gen_naive = np.array([s @ Wg @ H[j] for j in range(T)])
check_allclose('Luong general 向量化 vs 循环', a_gen, softmax(e_gen_naive))
print('注意：dot 与 general 一般给出【不同】的注意力分布（W 改变了几何）')
assert not np.allclose(a_dot, a_gen)
print('✅ Luong dot/general：均与朴素循环对拍一致；dot 即 Transformer QKᵀ 的雏形')

## 4 · 为什么 Transformer 要除以 √d_k

点积 `s·h` 的方差随维度 `d` 线性增长；维度大时 logit 过大，softmax 饱和（一个权重≈1 其余≈0），梯度趋零。

除以 `√d` 把方差拉回 ~1。下面用蒙特卡洛验证这个方差缩放，并看它对注意力「尖锐度」的影响。

In [ ]:
def dot_variance(d, n=20000):
    a = rng.standard_normal((n, d)); b = rng.standard_normal((n, d))
    dots = np.sum(a * b, axis=1)              # n 个 d 维随机向量点积
    return dots.var()

for d in [4, 16, 64, 256]:
    var = dot_variance(d)
    print(f'd={d:4d}: Var(s·h)={var:7.2f}  (≈d)   Var 缩放后(÷√d)≈{var/d:.2f}')
    assert abs(var - d) / d < 0.1, '点积方差应≈d'
# 缩放对 softmax 尖锐度的影响：同一组能量，÷√d 后分布更平缓
d = 64
e = rng.standard_normal(8) * np.sqrt(d)        # 模拟 d 维点积量级的能量
sharp = softmax(e); mild = softmax(e / np.sqrt(d))
print(f'\n未缩放 max 权重={sharp.max():.3f}（饱和）  缩放后 max 权重={mild.max():.3f}（平缓）')
assert sharp.max() > mild.max(), '不缩放更易饱和到接近 one-hot'
print('✅ 验证：Var(点积)≈d；÷√d 防止 softmax 饱和、保住梯度（Transformer 的关键细节）')

## 5 · 对齐矩阵：堆叠每步注意力权重

把每个 decoder 步的权重行 `α_{t,:}` 按 t 堆起来 → 对齐矩阵 `A`(T_target, T_source)。
**每一行（一个解码步）和为 1**。下面对一串 decoder 状态批量算 Bahdanau 注意力，组装对齐矩阵。

In [ ]:
def alignment_matrix(S, H, v, Ws, Wh):
    '''S:(T_tgt, D_s) 一串 decoder 状态；返回 A:(T_tgt, T_src) 对齐矩阵, C:(T_tgt,D_e) 各步上下文。'''
    T_tgt = S.shape[0]; T_src, D_e = H.shape
    A = np.zeros((T_tgt, T_src)); C = np.zeros((T_tgt, D_e))
    for t in range(T_tgt):
        C[t], A[t] = bahdanau_attention(S[t], H, v, Ws, Wh)
    return A, C

T_tgt = 3
S = rng.standard_normal((T_tgt, D_s))
A, C = alignment_matrix(S, H, v, Ws, Wh)
print('对齐矩阵 A 形状 =', A.shape, '(目标×源)')
print('每行和 =', np.round(A.sum(axis=1), 6))
assert A.shape == (T_tgt, T) and C.shape == (T_tgt, D_e)
assert np.allclose(A.sum(axis=1), 1.0), '对齐矩阵每行(每个解码步)应和为 1'
print('✅ 对齐矩阵：行和恒为 1；其形状将编码源-目标对应结构（胶囊里看对角线）')

## 6 · 注意力是可微的：对打分做数值梯度检验

注意力能端到端训练，全靠 softmax 把「硬选择」变成「可微加权」。

验证：上下文向量某分量对 decoder 状态 `s` 的梯度，解析（自己推）与数值（差分）应吻合。这里用 **dot** 注意力，便于手推解析梯度。

In [ ]:
# 用 dot 注意力：c = softmax(H s) @ H。验证 ∂(c[k]) / ∂s 可微且数值一致。
k = 2                                   # 看上下文第 k 个分量
def context_k(s_vec):
    e = H @ s_vec
    a = softmax(e)
    return (a @ H)[k]

s_test = rng.standard_normal(D_e)
g_num = numerical_grad(context_k, s_test.copy())
# 解析梯度：c_k = Σ_j a_j H[j,k]; a=softmax(He). dc_k/ds = Σ_j H[j,k] * da_j/ds
# da_j/ds = a_j (H[j] - Σ_m a_m H[m])  (softmax×线性 的标准结果)
e = H @ s_test; a = softmax(e); Hbar = a @ H
g_ana = sum(a[j] * H[j, k] * (H[j] - Hbar) for j in range(T))
err = rel_error(g_num, g_ana)
print(f'∂context[{k}]/∂s  解析 vs 数值  相对误差 = {err:.2e}')
assert err < 1e-5, '注意力对打分应可微且梯度正确'
print('✅ 注意力可微：softmax 把离散选择变连续加权，故可端到端梯度训练')

## 7 · teacher forcing vs 贪心解码

decoder 自回归生成。**训练（teacher forcing）**：喂真实上一词；**推理（free-running）**：喂模型自己的预测（贪心=取 argmax）。

下面用一个**固定的玩具 decoder 概率函数**模拟两种模式，展示它们产生不同的输入流，并实现带停止条件的贪心解码。

In [ ]:
# 玩具：词表 {0:'<bos>',1:'a',2:'b',3:'c',4:'<eos>'}；decoder 给定上一词→下一词分布(固定矩阵)
VOCAB = ['<bos>', 'a', 'b', 'c', '<eos>']
BOS, EOS = 0, 4
# 转移打分（行=上一词，列=下一词）；这是一个固定、确定性的玩具'模型'
LOGITS = np.array([
    [-9, 2.0, 1.0, 0.5, -9],   # <bos> -> 多半 'a'
    [-9, 0.1, 2.0, 0.5, 0.2],  # a     -> 多半 'b'
    [-9, 0.1, 0.1, 2.0, 0.3],  # b     -> 多半 'c'
    [-9, 0.1, 0.1, 0.1, 2.0],  # c     -> 多半 <eos>
    [-9, -9, -9, -9, 9.0],     # <eos> -> <eos>
])
def next_dist(prev_token):
    return softmax(LOGITS[prev_token])

def greedy_decode(max_len=10):
    '''自由运行：每步取 argmax 喂回，遇 <eos> 或超长停止。'''
    out = []
    prev = BOS
    for _ in range(max_len):
        nxt = int(np.argmax(next_dist(prev)))
        out.append(nxt)
        if nxt == EOS:
            break
        prev = nxt
    return out

gen = greedy_decode()
print('贪心解码输出:', [VOCAB[i] for i in gen])
assert gen[-1] == EOS, '贪心解码必须在 <eos> 停止'
assert [VOCAB[i] for i in gen] == ['a','b','c','<eos>']
print('✅ 贪心解码：argmax 自由运行，带停止条件，得到 a b c <eos>')

In [ ]:
# teacher forcing vs free-running 的输入流差异
target = [1, 2, 3, EOS]            # 真实目标 a b c <eos>
# teacher forcing：每步输入 = 真实上一词 [<bos>, a, b, c]
tf_inputs = [BOS] + target[:-1]
# free-running：每步输入 = 模型自己上一步 argmax
fr_inputs, prev = [BOS], BOS
for _ in range(len(target) - 1):
    prev = int(np.argmax(next_dist(prev)))
    fr_inputs.append(prev)
print('teacher forcing 输入流:', [VOCAB[i] for i in tf_inputs])
print('free-running   输入流:', [VOCAB[i] for i in fr_inputs])
print('本例两者恰好一致（玩具模型很准）；但训练用真实词、推理用预测词，机制不同 → exposure bias')
# 验证错位关系：teacher forcing 的 (输入, 标签) 错开一格
labels = target                   # [a, b, c, <eos>]
assert len(tf_inputs) == len(labels)
assert tf_inputs[1:] == labels[:-1], 'TF: 输入[t+1]==标签[t]（错位一格）'
print('✅ teacher forcing 的输入是标签右移一格（前补 <bos>）—— loss 对齐的关键')

---
## ✏️ 练习 1：从零实现加性 + 乘性打分函数

实现 `attention_scores(s, H, kind, params)`，支持三种 `kind`：
- `'dot'`：`s·h_j`（params 不用）；
- `'general'`：`sᵀ W h_j`（params={'W':W}）；
- `'additive'`：`vᵀ tanh(W_s s + W_h h_j)`（params={'v':v,'Ws':Ws,'Wh':Wh}）。

返回**未归一化的能量** `e`(T,)。

In [ ]:
def attention_scores(s, H, kind, params=None):
    params = params or {}
    T = H.shape[0]
    # TODO: 按 kind 分别计算能量 e (T,)：
    #   'dot'      -> H @ s
    #   'general'  -> (H @ params['W'].T) @ s
    #   'additive' -> tanh(Ws@s + H@Wh.T) @ v
    raise NotImplementedError
    return e

In [ ]:
# —— 练习 1 自测 ——
e_dot = attention_scores(s, H, 'dot')
e_gen = attention_scores(s, H, 'general', {'W': Wg})
e_add = attention_scores(s, H, 'additive', {'v': v, 'Ws': Ws, 'Wh': Wh})
assert e_dot.shape == (T,) and e_gen.shape == (T,) and e_add.shape == (T,)
check_allclose('练习1 dot',      e_dot, np.array([s @ H[j] for j in range(T)]))
check_allclose('练习1 general',  e_gen, np.array([s @ Wg @ H[j] for j in range(T)]))
check_allclose('练习1 additive', e_add, np.array([v @ np.tanh(Ws@s + Wh@H[j]) for j in range(T)]))
print('✅ 练习 1 通过：三种打分函数均与朴素循环一致')

## ✏️ 练习 2：组装对齐矩阵并验证行和

给定一串 decoder 状态 `S`(T_tgt, D_s)、encoder 状态 `H`，用 **dot** 打分，实现 `build_alignment(S, H)` 返回对齐矩阵 `A`(T_tgt, T_src)（每行 softmax 后的注意力权重）。

In [ ]:
def build_alignment(S, H):
    # TODO: 对每个 t，用 dot 打分 e=H@S[t]，softmax 得权重，堆成 A
    #       （提示：可一次性 E = S @ H.T，再对每行 softmax）
    raise NotImplementedError
    return A

In [ ]:
# —— 练习 2 自测 ——
S2 = rng.standard_normal((4, D_e))     # dot 需 D_s==D_e
A2 = build_alignment(S2, H)
assert A2.shape == (4, T), '对齐矩阵形状应为 (T_tgt, T_src)'
assert np.allclose(A2.sum(axis=1), 1.0), '每行应和为 1'
assert (A2 >= 0).all(), '权重应非负'
# 与逐行 dot 注意力对拍
A2_ref = np.stack([luong_attention(S2[t], H)[1] for t in range(4)])
check_allclose('练习2 对齐矩阵', A2, A2_ref)
print('✅ 练习 2 通过：对齐矩阵每行和=1，且与逐步注意力一致')

## ✏️ 练习 3：贪心解码（带停止条件）

给定一个 `step_fn(prev_token) -> 概率分布`、起始 `bos`、终止 `eos`、最大长度 `max_len`，实现 `greedy(step_fn, bos, eos, max_len)`：每步取 argmax，遇 eos 或超长停止，返回生成的 token 列表（**不含 bos，含 eos**）。

In [ ]:
def greedy(step_fn, bos, eos, max_len=20):
    out = []
    prev = bos
    # TODO: 循环 max_len 次：算分布→argmax→append；是 eos 就 break；否则 prev=nxt
    raise NotImplementedError
    return out

In [ ]:
# —— 练习 3 自测 ——
g = greedy(next_dist, BOS, EOS, max_len=10)
assert g[-1] == EOS, '必须以 eos 结束'
assert [VOCAB[i] for i in g] == ['a', 'b', 'c', '<eos>']
# 超长保护：一个永不输出 eos 的 step_fn 也必须在 max_len 停
never_eos = lambda prev: softmax(np.array([0., 5., 0., 0., -9.]))  # 总选 'a'
g2 = greedy(never_eos, BOS, EOS, max_len=6)
assert len(g2) == 6 and EOS not in g2, '永不 eos 时应在 max_len 截断'
print('✅ 练习 3 通过：贪心解码正确处理 eos 与 max_len 两种停止条件')

## ✏️ 练习 4：teacher forcing 的标签错位

teacher forcing 训练时，给定真实目标 `target`（含末尾 eos），
实现 `tf_pair(target, bos)` 返回 `(decoder_inputs, labels)`：
- `decoder_inputs` = `[bos] + target[:-1]`（右移一格，前补 bos）；
- `labels` = `target`（decoder 每步要预测的下一个词）。

In [ ]:
def tf_pair(target, bos):
    # TODO: 返回 (decoder_inputs, labels)，二者等长且错位一格
    raise NotImplementedError
    return decoder_inputs, labels

In [ ]:
# —— 练习 4 自测 ——
tgt = [1, 2, 3, EOS]
di, lb = tf_pair(tgt, BOS)
assert di == [BOS, 1, 2, 3] and lb == [1, 2, 3, EOS]
assert len(di) == len(lb), '输入与标签等长'
assert di[1:] == lb[:-1], '输入右移一格 == 标签左移一格'
print('decoder 输入:', [VOCAB[i] for i in di])
print('预测标签   :', [VOCAB[i] for i in lb])
print('✅ 练习 4 通过：teacher forcing 的输入=标签右移一格（前补 bos）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def attention_scores(s, H, kind, params=None):
    params = params or {}
    if kind == 'dot':
        e = H @ s
    elif kind == 'general':
        e = (H @ params['W'].T) @ s
    elif kind == 'additive':
        e = np.tanh(params['Ws'] @ s + H @ params['Wh'].T) @ params['v']
    else:
        raise ValueError(kind)
    return e

In [ ]:
# 练习 2 参考答案
def build_alignment(S, H):
    E = S @ H.T                      # (T_tgt, T_src) 一次矩阵乘算全部 dot 打分
    return softmax(E, axis=1)        # 每行 softmax

In [ ]:
# 练习 3 参考答案
def greedy(step_fn, bos, eos, max_len=20):
    out, prev = [], bos
    for _ in range(max_len):
        nxt = int(np.argmax(step_fn(prev)))
        out.append(nxt)
        if nxt == eos:
            break
        prev = nxt
    return out

In [ ]:
# 练习 4 参考答案
def tf_pair(target, bos):
    decoder_inputs = [bos] + list(target[:-1])
    labels = list(target)
    return decoder_inputs, labels

---
## 🧪 真实数据胶囊：在【复制 / 反转】任务上训练注意力，看对齐矩阵

用**固定随机种子**生成真实可复现的 token 序列，做两个经典玩具任务：
- **复制（copy）**：目标 = 源 → 正确对齐应是**主对角线**；
- **反转（reverse）**：目标 = 源逆序 → 正确对齐应是**反对角线**。

我们**直接训练一个 general 注意力的打分矩阵 W**（用最简单的、可解析梯度的目标）让注意力学出正确对齐，再用对齐矩阵的**对角性指标**客观验证「注意力学到了对齐」——不靠肉眼。

> 这正是 Bahdanau 论文里「对齐热图」的玩具复刻：可解释性 = 看模型在每步关注源的哪里。

In [ ]:
# 固定可复现的合成序列：每个'词'用一个唯一的随机嵌入向量表示
cap_rng = np.random.default_rng(20240615)
L, D = 6, 8                                  # 序列长 6，嵌入维 8
EMB = cap_rng.standard_normal((L, D))        # 第 i 个位置的源词嵌入（互不相同）
# encoder 状态直接用嵌入（玩具：跳过 RNN，聚焦注意力对齐本身）
H_src = EMB.copy()                           # (L, D) 源
# 任务定义：目标位置 t 应对齐到哪个源位置
def target_align(task):
    if task == 'copy':    return np.arange(L)        # t -> t
    if task == 'reverse': return np.arange(L)[::-1]  # t -> L-1-t
    raise ValueError(task)

# decoder 在第 t 步的 query：用'目标词的嵌入'（teacher forcing：已知目标）
# copy 的目标词嵌入 = EMB[t]；reverse 的 = EMB[L-1-t]
def target_queries(task):
    return EMB[target_align(task)]               # (L, D) 每个解码步的 query

print('源嵌入 EMB 形状:', EMB.shape)
print('copy 任务对齐目标   :', target_align('copy'))
print('reverse 任务对齐目标:', target_align('reverse'))
assert list(target_align('reverse')) == [5,4,3,2,1,0]
print('✅ 合成序列就绪（固定种子，完全可复现）')

In [ ]:
def diag_score(A, target_idx):
    '''对齐质量：每个解码步 argmax 命中目标源位置的比例（1.0=完美对齐）。'''
    pred = A.argmax(axis=1)
    return float(np.mean(pred == target_idx))

def attention_general(S, H, W):
    '''general 注意力的对齐矩阵：E = S W Hᵀ 再逐行 softmax。'''
    E = (S @ W) @ H.T
    return softmax(E, axis=1)

# 未训练（W=0）时注意力是均匀的，对齐质量≈随机
W0 = np.zeros((D, D))
A_copy0 = attention_general(target_queries('copy'), H_src, W0)
print('未训练 copy 对齐质量 =', diag_score(A_copy0, target_align('copy')), '（均匀，≈随机）')
assert np.allclose(A_copy0, 1.0 / L), 'W=0 时应是均匀注意力'
print('✅ 基线确认：未训练时注意力均匀、对齐无结构')

**🧪 胶囊练习**：训练 general 注意力的 `W`，让对齐矩阵收敛到正确结构。

用最简单可解析的目标：希望「源 query 与正确源 key 的打分尽量高」。对 copy/reverse，正确 key 就是源位置 `target_align(task)`。实现 `train_alignment(task, lr, steps)`：对每个解码步，最大化正确源位置的注意力 log 概率（即最小化交叉熵 `-log α[t, target[t]]`），梯度上升更新 `W`。

提示：交叉熵对能量 `E` 的梯度是 `α - onehot(target)`（softmax+CE 的标准结果），再链式回传到 `W`（`E = S W Hᵀ` → `dW = Sᵀ (dE) H`）。

In [ ]:
def train_alignment(task, lr=0.5, steps=300):
    S = target_queries(task)                 # (L, D) 各步 query
    tgt = target_align(task)                 # (L,) 正确源位置
    W = np.zeros((D, D))
    onehot = np.eye(L)[tgt]                   # (L, L) 每步的目标分布
    for _ in range(steps):
        # TODO: 前向 E=(S@W)@H_src.T (L,L)；A=softmax(E,axis=1)
        #       梯度 dE = (A - onehot)/L  （交叉熵对 logits）
        #       dW = S.T @ dE @ H_src     （链式：E=S W H^T）
        #       W -= lr * dW              （最小化交叉熵）
        raise NotImplementedError
    A = softmax((S @ W) @ H_src.T, axis=1)
    return W, A

In [ ]:
# 🧪 胶囊自测
for task in ['copy', 'reverse']:
    W, A = train_alignment(task, lr=0.5, steps=300)
    q = diag_score(A, target_align(task))
    print(f'{task:8s} 训练后对齐质量 = {q:.2f}  | 对齐矩阵 argmax = {A.argmax(1)}')
    assert q == 1.0, f'{task} 注意力应学出完美对齐'
    assert np.allclose(A.sum(1), 1.0)
# 复制任务应呈主对角线、反转应呈反对角线
_, A_copy = train_alignment('copy', steps=300)
_, A_rev  = train_alignment('reverse', steps=300)
assert list(A_copy.argmax(1)) == [0,1,2,3,4,5], 'copy 应是主对角线'
assert list(A_rev.argmax(1))  == [5,4,3,2,1,0], 'reverse 应是反对角线'
print('\n✅ 胶囊通过：注意力在 copy/reverse 上分别学出【对角/反对角】对齐——看见模型在每步关注哪里')

In [ ]:
# 📖 胶囊参考答案
def train_alignment(task, lr=0.5, steps=300):
    S = target_queries(task)
    tgt = target_align(task)
    W = np.zeros((D, D))
    onehot = np.eye(L)[tgt]
    for _ in range(steps):
        E = (S @ W) @ H_src.T            # (L,L)
        A = softmax(E, axis=1)
        dE = (A - onehot) / L           # 交叉熵对 logits 的梯度
        dW = S.T @ dE @ H_src           # 链式回传到 W
        W -= lr * dW
    A = softmax((S @ W) @ H_src.T, axis=1)
    return W, A

---
## 🔧 旁注：你刚写的，就是 Transformer 注意力的祖先

对照一下你实现的与 Transformer 的缩放点积注意力：

```python
# 你的 Luong dot（本 notebook 第 3 节）
E = H @ s            # 打分：query·key
A = softmax(E)       # 归一化
context = A @ H      # 加权求和

# Transformer（Vaswani 2017）—— 只多了 √d 缩放、批量 Q、自注意力
E = Q @ K.T / np.sqrt(d_k)   # 缩放点积（第 4 节验证了为何要 √d）
A = softmax(E, axis=-1)
context = A @ V
```
差别只在：① query 从单个 `s` 变成一批 `Q`；② key/value 从 encoder 变成**同一序列自己**（self-attention）；③ 多头并行；④ 除以 √d。
**机制内核完全一致**——你已经从零写过它了。

### 小结
- **seq2seq** = encoder 压源 + decoder 自回归生成；唯一桥梁是 context vector。
- **定长瓶颈**：固定维度 context 装不下长句 → 长句性能悬崖。
- **注意力** = 打分→softmax→加权求和，让 decoder 每步**动态回看**源；context 随步变化，瓶颈消失。
- **加性(Bahdanau) vs 乘性(Luong dot/general)**：后者是矩阵乘、并行友好，dot+√d 即 **Transformer** 注意力。
- **对齐矩阵**：每行和=1，形状泄露源-目标对应（copy→对角、reverse→反对角）。
- **teacher forcing / exposure bias / 贪心解码**：训练喂真实词、推理喂预测词的鸿沟与解码策略。

🎓 **本课终点**：你已备齐 Transformer 的全部零件（注意力=本模块、残差=模块02、LayerNorm=模块02/04）。下一站 → **C1 · Transformer**。